In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# imports
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error
import numpy as np



In [ ]:
df = pd.read_csv('/kaggle/input/q1-ka-ai-2026/Q1_data.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Delievery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write df code here:
df = df.drop("Order_ID", axis = 1)

In [ ]:
# Task 2: Write your code here:

# Since missing samples are relatively not that much, im gonna drop them

df = df.dropna()

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

categorical_cols = ["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type"]

for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

In [ ]:
df.info()

In [ ]:
# Task 5: Write your code here:

scaler = StandardScaler()

# i will not be scaling cols that have been encoded and the target varible
cols_no_target_or_encoded = ["Distance_km", "Preparation_Time_min", "Courier_Experience_yrs" ]


df[cols_no_target_or_encoded] = scaler.fit_transform(df[cols_no_target_or_encoded])

In [ ]:
# Task 6: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(df["Delivery_Time"].value_counts(), bins=50, edgecolor='black')
plt.title('Distribution')
plt.xlabel('Delievery Time')
plt.ylabel('Frequency')
plt.show()

# there is an imbalance so doing stratified sampling here would be much better

In [ ]:
X = df.drop("Delivery_Time", axis=1).copy()
y = df["Delivery_Time"]



In [ ]:
# Task 2,3,4,5: Write your code here:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for test, remaining 80% for train
    random_state=42,      # reproducible output
    shuffle=True,         # representative splits
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


model = RandomForestClassifier(
      n_estimators=200,
      max_depth=10
  )
mae = []
mae_sum = 0
y_preds = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate
  mae_sum+=mean_absolute_error(y_test, y_pred)
  y_preds.append(y_pred)




print("MAE:", mae_sum)
print("avg MAE:", mae_sum/5)



In [ ]:
model.feature_importances_

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.hist(y_preds)
plt.show()

In [ ]:
# Task Bonus: Write your code here:
